In [1]:
import pandas as pd
import numpy as np

In [2]:
TRANSACTIONS_PATH = "/Users/kamilaya/Desktop/thesis/transactions_train.csv"
ARTICLES_PATH     = "/Users/kamilaya/Desktop/thesis/articles.csv"
MACRO_PATH        = "/Users/kamilaya/Desktop/thesis/eurostat_macro_clean.csv"   
OUTPUT_PATH       = "tshirts_final_final.csv"

In [3]:
print("Step 1: Cleaning macro data...")
macro_raw = pd.read_csv(MACRO_PATH)
print(f"  Raw macro shape: {macro_raw.shape}  ({macro_raw['year_month'].nunique()} months × {len(macro_raw)//macro_raw['year_month'].nunique()} rows/month)")
 
macro = (
    macro_raw
    .groupby("year_month")[["eurozone_hicp", "eurozone_unemployment_rate", "eurozone_cci"]]
    .mean()
    .reset_index()
)
print(f"  Clean macro shape: {macro.shape}")
print(macro.to_string())

Step 1: Cleaning macro data...
  Raw macro shape: (27000, 4)  (25 months × 1080 rows/month)
  Clean macro shape: (25, 4)
   year_month  eurozone_hicp  eurozone_unemployment_rate  eurozone_cci
0     2018-09           1.56                   10.844444      0.583333
1     2018-10           1.44                   10.955556      0.700000
2     2018-11           1.24                   10.622222     -0.300000
3     2018-12           1.10                   10.461111     -2.308333
4     2019-01           1.30                   10.822222     -1.633333
5     2019-02           1.72                   10.716667     -1.575000
6     2019-03           1.42                   10.522222     -0.766667
7     2019-04           1.42                   10.344444     -2.775000
8     2019-05           1.34                   10.177778     -0.358333
9     2019-06           1.46                    9.888889     -0.625000
10    2019-07           1.48                    9.977778     -0.908333
11    2019-08           1.5

In [4]:
print("\nStep 2: Filtering articles to T-shirts...")
articles_raw = pd.read_csv(ARTICLES_PATH, dtype={"article_id": str})


Step 2: Filtering articles to T-shirts...


In [5]:
print("  Sample product_type_name values:", articles_raw["product_type_name"].value_counts().head(10).to_dict())
 
tshirt_articles = articles_raw[articles_raw["product_type_name"] == "T-shirt"].copy()

  Sample product_type_name values: {'Trousers': 11169, 'Dress': 10362, 'Sweater': 9302, 'T-shirt': 7904, 'Top': 4155, 'Blouse': 3979, 'Jacket': 3940, 'Shorts': 3939, 'Shirt': 3405, 'Vest top': 2991}


In [6]:
tshirt_articles = tshirt_articles[[
    "article_id",
    "product_type_name",
    "product_group_name",
    "graphical_appearance_name",
    "colour_group_name",
    "perceived_colour_value_name",
    "index_group_name",        # Ladieswear / Baby/Children / Divided etc.
]].drop_duplicates(subset=["article_id"])

In [7]:
print(f"  T-shirt articles found: {len(tshirt_articles):,}")
print(f"  index_group breakdown:\n{tshirt_articles['index_group_name'].value_counts().to_string()}")

  T-shirt articles found: 7,904
  index_group breakdown:
index_group_name
Baby/Children    3400
Menswear         1508
Ladieswear       1408
Divided          1082
Sport             506


In [8]:
print("\nStep 3: Building weekly sales from transactions (chunked)...")
 
tshirt_ids = set(tshirt_articles["article_id"].astype(str))
CHUNK_SIZE = 500_000
sales_chunks = []
chunk_num = 0


Step 3: Building weekly sales from transactions (chunked)...


In [9]:
for chunk in pd.read_csv(TRANSACTIONS_PATH, chunksize=CHUNK_SIZE,
                          dtype={"article_id": str, "customer_id": str}):
    chunk_num += 1
    if chunk_num % 10 == 0:
        print(f"  ...processed {chunk_num * CHUNK_SIZE:,} transaction rows")
 
    # Keep only T-shirt transactions
    chunk = chunk[chunk["article_id"].isin(tshirt_ids)].copy()
    if len(chunk) == 0:
        continue
 
    # Parse date and snap to week start (Monday)
    chunk["t_dat"] = pd.to_datetime(chunk["t_dat"])
    chunk["week_start"] = chunk["t_dat"] - pd.to_timedelta(chunk["t_dat"].dt.dayofweek, unit="D")
    chunk["week_start"] = chunk["week_start"].dt.date   # keep as date, not datetime
 
    sales_chunks.append(chunk[["article_id", "week_start", "price"]])
 
print(f"  Done. Total chunks processed: {chunk_num}")

  ...processed 5,000,000 transaction rows
  ...processed 10,000,000 transaction rows
  ...processed 15,000,000 transaction rows
  ...processed 20,000,000 transaction rows
  ...processed 25,000,000 transaction rows
  ...processed 30,000,000 transaction rows
  Done. Total chunks processed: 64


In [10]:
# Concatenate all T-shirt transactions
txn = pd.concat(sales_chunks, ignore_index=True)
print(f"  T-shirt transaction rows: {len(txn):,}")

  T-shirt transaction rows: 2,203,750


In [11]:
weekly_sales = (
    txn
    .groupby(["article_id", "week_start"])
    .agg(
        weekly_sales_volume=("price", "count"),
        avg_weekly_price=("price", "mean")
    )
    .reset_index()
)
print(f"  Unique (article, week) rows: {len(weekly_sales):,}")
print(f"  Date range: {weekly_sales['week_start'].min()}  →  {weekly_sales['week_start'].max()}")

  Unique (article, week) rows: 151,985
  Date range: 2018-09-17  →  2020-09-21


In [12]:
weekly_sales["week_start"] = pd.to_datetime(weekly_sales["week_start"])
weekly_sales["year_month"] = weekly_sales["week_start"].dt.to_period("M").astype(str)

In [13]:
print("\nStep 4: Merging datasets...")
 
# 4a. Add product attributes
df = weekly_sales.merge(tshirt_articles, on="article_id", how="left")
print(f"  After adding product attributes: {df.shape}")


Step 4: Merging datasets...
  After adding product attributes: (151985, 11)


In [14]:
df = df.merge(macro, on="year_month", how="left")
print(f"  After adding macro data: {df.shape}")

  After adding macro data: (151985, 14)


In [15]:
assert len(df) == len(weekly_sales), "Row count changed after merge — check for duplicate keys in macro or articles!"
print("  ✓ Row count unchanged — merge was clean (no Cartesian join).")

  ✓ Row count unchanged — merge was clean (no Cartesian join).


In [16]:
print("\nStep 5: Feature engineering...")
df = df.sort_values(["article_id", "week_start"]).reset_index(drop=True)


Step 5: Feature engineering...


In [17]:
df["week_of_year"] = df["week_start"].dt.isocalendar().week.astype(int)
df["month"]        = df["week_start"].dt.month
df["quarter"]      = df["week_start"].dt.quarter
df["year"]         = df["week_start"].dt.year

In [18]:
df["is_summer"] = df["month"].isin([5, 6, 7, 8]).astype(int)

In [19]:
first_sale = df.groupby("article_id")["week_start"].min().rename("first_sale_week")
df = df.merge(first_sale, on="article_id", how="left")
df["product_age_weeks"] = ((df["week_start"] - df["first_sale_week"]).dt.days // 7)
df.drop(columns=["first_sale_week"], inplace=True)

In [20]:
df["sales_lag1"] = df.groupby("article_id")["weekly_sales_volume"].shift(1)
df["sales_lag2"] = df.groupby("article_id")["weekly_sales_volume"].shift(2)

In [21]:
df["sales_rolling4"] = (
    df.groupby("article_id")["weekly_sales_volume"]
    .transform(lambda x: x.shift(1).rolling(window=4, min_periods=1).mean())
)

In [22]:
df["real_price"] = df["avg_weekly_price"] / (1 + df["eurozone_hicp"] / 100)

In [23]:
print("\nStep 6: Final quality checks...")
print(f"\n  Final dataset shape: {df.shape}")
print(f"\n  Columns: {df.columns.tolist()}")
print(f"\n  Null counts:\n{df.isnull().sum().to_string()}")
print(f"\n  Sales volume distribution:\n{df['weekly_sales_volume'].describe().to_string()}")
print(f"\n  Macro coverage:\n{df[['eurozone_hicp','eurozone_unemployment_rate','eurozone_cci']].describe().to_string()}")


Step 6: Final quality checks...

  Final dataset shape: (151985, 24)

  Columns: ['article_id', 'week_start', 'weekly_sales_volume', 'avg_weekly_price', 'year_month', 'product_type_name', 'product_group_name', 'graphical_appearance_name', 'colour_group_name', 'perceived_colour_value_name', 'index_group_name', 'eurozone_hicp', 'eurozone_unemployment_rate', 'eurozone_cci', 'week_of_year', 'month', 'quarter', 'year', 'is_summer', 'product_age_weeks', 'sales_lag1', 'sales_lag2', 'sales_rolling4', 'real_price']

  Null counts:
article_id                         0
week_start                         0
weekly_sales_volume                0
avg_weekly_price                   0
year_month                         0
product_type_name                  0
product_group_name                 0
graphical_appearance_name          0
colour_group_name                  0
perceived_colour_value_name        0
index_group_name                   0
eurozone_hicp                      0
eurozone_unemployment_rate 

In [24]:
weeks_per_article = df.groupby("article_id")["week_start"].count()
cold_start = (weeks_per_article <= 3).sum()
print(f"\n  Cold-start articles (≤3 weeks of sales): {cold_start:,} ({cold_start/len(weeks_per_article)*100:.1f}%)")


  Cold-start articles (≤3 weeks of sales): 1,208 (15.3%)


In [25]:
df.to_csv(OUTPUT_PATH, index=False)
print(f"\n  ✓ Saved to: {OUTPUT_PATH}")
print("\nDone!")


  ✓ Saved to: tshirts_final_final.csv

Done!


In [26]:
df.head()

,article_id,week_start,weekly_sales_volume,avg_weekly_price,year_month,product_type_name,product_group_name,graphical_appearance_name,colour_group_name,perceived_colour_value_name,...,week_of_year,month,quarter,year,is_summer,product_age_weeks,sales_lag1,sales_lag2,sales_rolling4,real_price
0,0189955076,2018-09-24,1,0.010153,2018-09,T-shirt,Garment Upper body,Melange,Light Blue,Dusty Light,...,39,9,3,2018,0,0,NaN,NaN,NaN,0.009997
1,0189955076,2019-01-07,1,0.009136,2019-01,T-shirt,Garment Upper body,Melange,Light Blue,Dusty Light,...,2,1,1,2019,0,15,1.0,NaN,1.0,0.009018
2,0189955076,2019-02-25,1,0.009136,2019-02,T-shirt,Garment Upper body,Melange,Light Blue,Dusty Light,...,9,2,1,2019,0,22,1.0,1.0,1.0,0.008981
3,0189955076,2019-06-03,1,0.010153,2019-06,T-shirt,Garment Upper body,Melange,Light Blue,Dusty Light,...,23,6,2,2019,1,36,1.0,1.0,1.0,0.010006
4,0194902033,2018-09-24,2,0.005068,2018-09,T-shirt,Garment Upper body,Solid,Black,Dark,...,39,9,3,2018,0,0,NaN,NaN,NaN,0.004990
